conda install pytorch torchvision torchaudio pytorch-cuda=12.4 -c pytorch -c nvidia

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# 간단 실행 테스트
x = torch.randn(3, 3).to("cuda" if torch.cuda.is_available() else "cpu")
print("tensor device:", x.device)

In [ ]:
# ==================================================
# 0️⃣ 필수 라이브러리
# ==================================================
import os, math, time, base64
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from jinja2 import Template

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import shap

In [75]:
# -*- coding: utf-8 -*-
"""
🌱 완전 통합 파이프라인 (Plant → Region)
- CNN-BiLSTM 기반
- 1차 학습 / 조건부 2차 학습 (weight step)
- Region별 Fine-tuning
- Resume, AMP, EarlyStopping
- SHAP 안전 분석 (안전하게 시퀀스 처리)
- HTML/CSV 보고서 자동 생성 + Plant vs Region SHAP 비교
- 출력 강화: ETA, Best 표시, 상세 지표
- 음수 R² -> 0 보정 및 제외된 발전소 별도 표기
"""
import os, time, warnings, base64, webbrowser, json
from io import BytesIO
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from jinja2 import Template
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import shap

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {DEVICE}")

# ----------------------------
# === 사용자 설정 ===
# ----------------------------
TRAIN_CSV = r"C:\ESG_Project1\file\merge_data\train.csv"
VAL_CSV   = r"C:\ESG_Project1\file\merge_data\val.csv"
TEST_CSV  = r"C:\ESG_Project1\file\merge_data\test.csv"
SAVE_DIR  = r"C:\ESG_Project1\cnn_lstm\output"
CKPT_DIR  = os.path.join(SAVE_DIR,"checkpoints")
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

TIME_COL, GROUP_COL, REGION_COL, TARGET_COL = "일시", "발전구분", "지역", "합산발전량(MWh)"
WEATHER_COLS = [
    "기온(°C)", "강수량(mm)", "풍속(m/s)", "습도(%)", "증기압(hPa)",
    "일조(hr)", "일사(MJ/m2)", "적설(cm)", "전운량(10분위)", "중하층운량(10분위)"
]
TIME_FEATS = ["hour_sin", "hour_cos", "doy_sin", "doy_cos"]

SEQ_LEN, HORIZON = 168, 24
BATCH = 128
EPOCHS = 50
FINE_TUNE_EPOCHS = 10
LR = 1e-3
PATIENCE = 0
SHAP_SAMPLE = 50
VAL_R2_THRESHOLD = 0.5
SECONDARY_WEIGHT_STEPS = [1.0, 0.5, 0.3]
RESUME = True   # Resume 기능 사용 여부 (전체 파이프라인 재개 가능)

# ----------------------------
# === 데이터 로드 + 전처리 ===
# ----------------------------
def add_time_feats(df):
    df = df.copy()
    df.sort_values([GROUP_COL, TIME_COL], inplace=True)
    df["hour"] = df[TIME_COL].dt.hour
    df["doy"] = df[TIME_COL].dt.dayofyear
    df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)
    df["doy_sin"] = np.sin(2*np.pi*df["doy"]/365)
    df["doy_cos"] = np.cos(2*np.pi*df["doy"]/365)
    return df

def add_lag_diff(df, lags=[1,3,6,24]):
    df = df.copy()
    for lag in lags:
        df[f"lag_{lag}"] = df.groupby(GROUP_COL)[TARGET_COL].shift(lag)
        df[f"diff_{lag}"] = df[TARGET_COL] - df[f"lag_{lag}"]
    df.fillna(0, inplace=True)
    return df

# load
train_raw = add_lag_diff(add_time_feats(pd.read_csv(TRAIN_CSV, parse_dates=[TIME_COL])))
val_raw   = add_lag_diff(add_time_feats(pd.read_csv(VAL_CSV, parse_dates=[TIME_COL])))
test_raw  = add_lag_diff(add_time_feats(pd.read_csv(TEST_CSV, parse_dates=[TIME_COL])))

all_candidate_feats = WEATHER_COLS + TIME_FEATS + [f"lag_{l}" for l in [1,3,6,24]] + [f"diff_{l}" for l in [1,3,6,24]]
feature_cols = [c for c in all_candidate_feats if c in train_raw.columns]
print(f"✅ feature_cols ({len(feature_cols)}): {feature_cols}")

# ----------------------------
# === 지역별 스케일러 적용 ===
# ----------------------------
region_scalers = {}
for region, grp in train_raw.groupby(REGION_COL):
    valid_cols = [c for c in feature_cols if c in grp.columns]
    if len(valid_cols)==0: continue
    std = StandardScaler().fit(grp[valid_cols])
    mm  = MinMaxScaler().fit(std.transform(grp[valid_cols]))
    tsc = StandardScaler().fit(np.log1p(grp[[TARGET_COL]].clip(lower=0.0)+1e-8))
    region_scalers[region] = (std, mm, tsc)
    train_raw.loc[grp.index, valid_cols] = mm.transform(std.transform(grp[valid_cols]))
    train_raw.loc[grp.index, TARGET_COL] = tsc.transform(np.log1p(grp[[TARGET_COL]].clip(lower=0.0)+1e-8))

for df in (val_raw, test_raw):
    for region, grp in df.groupby(REGION_COL):
        valid_cols = [c for c in feature_cols if c in grp.columns]
        if region in region_scalers and len(valid_cols)>0:
            std, mm, tsc = region_scalers[region]
            df.loc[grp.index, valid_cols] = mm.transform(std.transform(grp[valid_cols]))
            df.loc[grp.index, TARGET_COL] = tsc.transform(np.log1p(grp[[TARGET_COL]].clip(lower=0.0)+1e-8))

# ----------------------------
# === Dataset & Model ===
# ----------------------------
class TimeSeriesSeqDataset(Dataset):
    def __init__(self, df, seq_len=SEQ_LEN, horizon=HORIZON, feature_cols=feature_cols):
        self.seq_len = seq_len
        self.horizon = horizon
        self.features = df[feature_cols].values.astype(np.float32)
        self.targets = df[TARGET_COL].values.astype(np.float32)
        self.n_windows = max(0, len(df) - seq_len - horizon + 1)

    def __len__(self):
        return self.n_windows

    def __getitem__(self, i):
        x = self.features[i:i + self.seq_len]
        # 단일 시점 예측이면 y도 스칼라로 반환
        if self.horizon == 1:
            y = self.targets[i + self.seq_len]
        else:
            y = self.targets[i + self.seq_len : i + self.seq_len + self.horizon]
        return torch.from_numpy(x), torch.from_numpy(np.atleast_1d(y))


class CNN_BiLSTM(nn.Module):
    def __init__(self, input_dim=len(feature_cols), hidden=128, num_layers=2, horizon=HORIZON):
        super().__init__()
        self.conv1 = nn.Conv1d(input_dim, 128, 3, padding=1)
        self.conv2 = nn.Conv1d(128, 64, 3, padding=1)
        self.lstm = nn.LSTM(64, hidden, num_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden*2, horizon)
        self.relu = nn.ReLU()
        self.norm = nn.LayerNorm(hidden*2)
    def forward(self, x):
        x = x.permute(0,2,1)            # (B, C, T)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = x.permute(0,2,1)            # (B, T, C)
        x,_ = self.lstm(x)
        x = self.norm(x[:,-1,:])
        return self.fc(x)

# ----------------------------
# === 유틸 함수 ===
# ----------------------------
def predict(model, df, features=feature_cols, seq_len=SEQ_LEN, horizon=HORIZON):
    """
    모델 예측 수행
    - df: 입력 데이터프레임
    - features: feature 컬럼 리스트
    - seq_len: 시퀀스 길이
    - horizon: 예측 시점
    반환:
    - preds: 모델 예측값 (1차원 배열)
    - trues: 실제 타깃값 (1차원 배열)
    """
    ds = TimeSeriesSeqDataset(df, seq_len=seq_len, horizon=horizon, feature_cols=features)
    if len(ds) == 0:
        return np.array([]), np.array([])

    loader = DataLoader(ds, batch_size=BATCH, shuffle=False)
    preds, trues = [], []

    model.eval()
    with torch.no_grad():
        for X, y_true in loader:
            X = X.to(DEVICE)
            y_pred = model(X)
            # horizon=1인 경우 shape 맞추기
            if y_pred.ndim > 1 and y_pred.shape[1] == 1:
                y_pred = y_pred[:, 0]
            preds.append(y_pred.cpu().numpy())
            trues.append(y_true.cpu().numpy())

    preds = np.concatenate(preds).reshape(-1) if preds else np.array([])
    trues = np.concatenate(trues).reshape(-1) if trues else np.array([])

    return preds, trues

def calc_metrics(y_true, y_pred):
    """
    R², RMSE, MAE 계산
    - y_true, y_pred 길이가 다르거나 비어있으면 nan 반환
    """
    if len(y_true) == 0 or len(y_pred) == 0 or len(y_true) != len(y_pred):
        return float("nan"), float("nan"), float("nan")

    try:
        r2 = r2_score(y_true, y_pred)
    except Exception:
        r2 = float("nan")

    try:
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
    except Exception:
        rmse = float("nan")

    try:
        mae = mean_absolute_error(y_true, y_pred)
    except Exception:
        mae = float("nan")

    return r2, rmse, mae

# ==================================================
# 🔹 CNN-BiLSTM Plant/Region 학습 + EMA/R² + Resume + 매핑
# ==================================================
# EMA util
def ema_update(ema_val, new_val, alpha=0.3):
    return alpha*new_val + (1-alpha)*ema_val if ema_val is not None else new_val

def save_checkpoint(ckpt_path, model, optimizer, epoch, best_r2, extra=None):
    payload = {
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict() if optimizer is not None else None,
        "epoch": epoch,
        "best_r2": best_r2
    }
    if extra:
        payload.update(extra)
    torch.save(payload, ckpt_path)

def load_checkpoint(model, optimizer, ckpt_path):
    if os.path.exists(ckpt_path):
        try:
            ckpt = torch.load(ckpt_path, map_location=DEVICE)
            st = ckpt.get("model_state", ckpt)
            model.load_state_dict(st)
            if optimizer is not None and ckpt.get("optimizer_state", None) is not None:
                optimizer.load_state_dict(ckpt["optimizer_state"])
            start_epoch = int(ckpt.get("epoch", 0)) + 1
            best_r2 = float(ckpt.get("best_r2", -np.inf))
            print(f"🔄 Checkpoint 로드: {ckpt_path} | resume epoch={start_epoch} | best_r2={best_r2:.4f}")
            return start_epoch, best_r2, ckpt
        except Exception as e:
            print(f"⚠ Checkpoint 로드 실패 ({ckpt_path}): {e}")
            return 1, -np.inf, None
    return 1, -np.inf, None

def safe_fmt(v):
    if v is None or (isinstance(v,float) and np.isnan(v)):
        return "-"
    return f"{v:.4f}" if isinstance(v,(int,float)) else str(v)

# =========================
# 1️⃣ Plant 학습 + Checkpoint + EMA
# =========================
results_list = []
plant_models = {}        # plant -> best checkpoint path
plant_metrics_1st = {}   # plant -> (val_r2, val_rmse, val_mae)

def train_plant(model, optimizer, criterion, train_loader, val_loader, ckpt_dir, plant):
    ckpt_last = os.path.join(ckpt_dir,f"{plant}_last.pt")
    ckpt_best = os.path.join(ckpt_dir,f"{plant}_best.pt")

    start_epoch, best_r2 = 1, -np.inf
    if RESUME:
        if os.path.exists(ckpt_best):
            start_epoch, best_r2, _ = load_checkpoint(model, optimizer, ckpt_best)
        elif os.path.exists(ckpt_last):
            start_epoch, best_r2, _ = load_checkpoint(model, optimizer, ckpt_last)

    r2_ema, early_counter = None, 0
    epoch_start_time = time.time()
    for epoch in range(start_epoch, EPOCHS+1):
        # Train
        model.train()
        train_losses = []
        for X,y in train_loader:
            X,y = X.to(DEVICE),y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X),y)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
        
        # Validation
        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for Xv,yv in val_loader:
                Xv,yv = Xv.to(DEVICE), yv.to(DEVICE)
                yp = model(Xv)
                val_preds.append(yp[:,0].cpu().numpy())
                val_true.append(yv[:,0].cpu().numpy())
        val_preds = np.concatenate(val_preds) if val_preds else np.array([])
        val_true = np.concatenate(val_true) if val_true else np.array([])

        val_r2, val_rmse, val_mae = calc_metrics(val_true,val_preds)
        train_loss_mean = np.mean(train_losses) if train_losses else float("nan")
        r2_ema = ema_update(r2_ema, val_r2)

        best_mark=""
        if r2_ema is not None and r2_ema > best_r2:
            best_r2 = r2_ema
            save_checkpoint(ckpt_best, model, optimizer, epoch, best_r2)
            best_mark="★Best★"
            early_counter=0
        else:
            early_counter+=1

        save_checkpoint(ckpt_last, model, optimizer, epoch, best_r2)

        avg_epoch_time=(time.time()-epoch_start_time)/max(1,epoch-start_epoch+1)
        eta_s = avg_epoch_time*(EPOCHS-epoch)
        eta_str=f"{eta_s:.1f}s" if eta_s<60 else f"{eta_s/60:.2f}m"

        print(f"Epoch {epoch}/{EPOCHS} | TrainLoss={train_loss_mean:.6f} | Val R2={val_r2:.4f} | EMA R2={r2_ema:.4f} {best_mark} | RMSE={val_rmse:.4f} | MAE={val_mae:.4f} | ETA={eta_str}")

        if early_counter>=PATIENCE:
            print(f"⏹ EarlyStopping Triggered for {plant}")
            break

    # 최종 평가
    ckpt_eval = ckpt_best if os.path.exists(ckpt_best) else ckpt_last
    ck = torch.load(ckpt_eval,map_location=DEVICE)
    model.load_state_dict(ck["model_state"])
    val_preds_list, val_true_list = [], []
    with torch.no_grad():
        for Xv, yv in val_loader:
            Xv,yv = Xv.to(DEVICE), yv.to(DEVICE)
            yp = model(Xv)
            val_preds_list.append(yp[:,0].cpu().numpy())
            val_true_list.append(yv[:,0].cpu().numpy())
    val_preds_final = np.concatenate(val_preds_list)
    val_true_final = np.concatenate(val_true_list)
    v_r2,v_rmse,v_mae = calc_metrics(val_true_final,val_preds_final)
    return ckpt_eval, (v_r2,v_rmse,v_mae)

# Plant 학습 루프
for plant, df_train in train_raw.groupby(GROUP_COL):
    print(f"\n🌱 Plant 학습 시작: {plant} | train size: {len(df_train)}")
    df_val = val_raw[val_raw[GROUP_COL]==plant]
    if len(df_val)==0:
        print("❌ Validation 데이터 없음, 건너뜀.")
        continue

    train_loader = DataLoader(TimeSeriesSeqDataset(df_train), batch_size=BATCH, shuffle=True)
    val_loader = DataLoader(TimeSeriesSeqDataset(df_val), batch_size=BATCH, shuffle=False)

    model = CNN_BiLSTM().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.MSELoss()

    plant_dir = os.path.join(CKPT_DIR,"plant",plant)
    os.makedirs(plant_dir, exist_ok=True)
    ckpt_eval, metrics = train_plant(model, optimizer, criterion, train_loader, val_loader, plant_dir, plant)

    plant_models[plant] = ckpt_eval
    plant_metrics_1st[plant] = metrics
    results_list.append({"plant":plant,"val_r2":metrics[0],"val_rmse":metrics[1],"val_mae":metrics[2]})
    print(f"✅ 최종 평가 완료 (Plant={plant}) | R2={metrics[0]:.4f} | RMSE={metrics[1]:.4f} | MAE={metrics[2]:.4f}")

# =========================
# 2️⃣ Region Fine-tuning (Resume + 최고 R² Weight Step + 강화 출력 + 상태 표시)
# =========================
region_plant_map = {}   # Region -> 포함 Plant 목록
region_models = {}      # Region -> 최종 checkpoint 경로
region_metrics = {}     # Region -> (R2, RMSE, MAE)
region_ckpt_status = {} # Region -> 최종 ckpt 상태 표시 ("1차 Only", "Region 최종", "2차 Step wX")

for region, df_region in train_raw.groupby(REGION_COL):
    print(f"\n🌿 Region Fine-tuning 시작: {region}")
    df_val_region = val_raw[val_raw[REGION_COL]==region]
    if len(df_val_region)==0:
        print("❌ Region validation 데이터 없음, 건너뜀.")
        continue

    region_dir = os.path.join(CKPT_DIR, "region", region)
    os.makedirs(region_dir, exist_ok=True)
    ckpt_last = os.path.join(region_dir, f"{region}_last.pt")
    ckpt_best = os.path.join(region_dir, f"{region}_best.pt")

    plants_in_region = df_region[GROUP_COL].unique().tolist()
    region_plant_map[region] = plants_in_region
    print(f"🔹 Region에 포함된 Plant: {plants_in_region}")

    # Plant별 최적 ckpt 불러오기 → Region 초기화
    model = CNN_BiLSTM().to(DEVICE)
    plant_ckpts = []
    for p in plants_in_region:
        plant_ckpt_path = plant_models.get(p)
        if plant_ckpt_path and os.path.exists(plant_ckpt_path):
            ck = torch.load(plant_ckpt_path, map_location=DEVICE)
            plant_ckpts.append(ck["model_state"])
    if plant_ckpts:
        avg_state = {k: torch.stack([sd[k].float() for sd in plant_ckpts],0).mean(0) for k in plant_ckpts[0].keys()}
        model.load_state_dict(avg_state)
        print("🔹 Plant 최적 ckpt 평균으로 Region 초기화 완료.")
        final_status = "1차 Only"
    else:
        print("⚠ Plant ckpt 없음 → Region 무작위 초기화.")
        final_status = "Region 최종"

    optimizer = optim.Adam(model.parameters(), lr=LR*0.5)
    criterion = nn.MSELoss()

    # Resume 체크
    start_epoch, best_r2 = 1, -np.inf
    if RESUME:
        if os.path.exists(ckpt_best):
            start_epoch, best_r2, _ = load_checkpoint(model, optimizer, ckpt_best)
            final_status = "Region 최종"
        elif os.path.exists(ckpt_last):
            start_epoch, best_r2, _ = load_checkpoint(model, optimizer, ckpt_last)
            final_status = "Region 최종"

    train_loader = DataLoader(TimeSeriesSeqDataset(df_region), batch_size=BATCH, shuffle=True)
    val_loader = DataLoader(TimeSeriesSeqDataset(df_val_region), batch_size=BATCH, shuffle=False)

    r2_ema, early_counter = None, 0
    epoch_start_time = time.time()

    # --- Region Fine-tuning Epoch Loop ---
    for epoch in range(start_epoch, FINE_TUNE_EPOCHS+1):
        model.train()
        train_losses = []
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        # Validation
        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for Xv, yv in val_loader:
                Xv, yv = Xv.to(DEVICE), yv.to(DEVICE)
                yp = model(Xv)
                val_preds.append(yp[:,0].cpu().numpy())
                val_true.append(yv[:,0].cpu().numpy())
        val_preds = np.concatenate(val_preds) if val_preds else np.array([])
        val_true = np.concatenate(val_true) if val_true else np.array([])

        r2, rmse, mae = calc_metrics(val_true,val_preds)
        train_loss_mean = np.mean(train_losses) if train_losses else float("nan")
        r2_ema = ema_update(r2_ema,r2)

        best_mark = ""
        if r2_ema > best_r2:
            best_r2 = r2_ema
            save_checkpoint(ckpt_best, model, optimizer, epoch, best_r2)
            best_mark = "★Best★"
            early_counter = 0
            final_status = "Region 최종"
        else:
            early_counter += 1

        save_checkpoint(ckpt_last, model, optimizer, epoch, best_r2)

        # --- 2차 Weight Step checkpoint 저장 (val R2<0.5인 Plant만) ---
        if SECONDARY_WEIGHT_STEPS:
            for p in plants_in_region:
                p_r2,_,_ = plant_metrics_1st.get(p,(0,0,0))
                if p_r2 < VAL_R2_THRESHOLD:
                    for w in SECONDARY_WEIGHT_STEPS:
                        weight_ckpt_path = os.path.join(region_dir,f"{p}_2nd_w{w}.pt")
                        save_checkpoint(weight_ckpt_path, model, optimizer, epoch, best_r2, extra={"weight_step":w})
                        print(f"💾 Plant={p} | Weight Step={w} | ckpt 저장 완료")

        avg_epoch_time = (time.time() - epoch_start_time) / max(1, epoch-start_epoch+1)
        eta_s = avg_epoch_time * max(0, FINE_TUNE_EPOCHS - epoch)
        eta_str = f"{eta_s:.1f}s" if eta_s < 60 else f"{eta_s/60:.2f}m"

        print(f"Epoch {epoch}/{FINE_TUNE_EPOCHS} | TrainLoss={train_loss_mean:.6f} | "
              f"Region R2={r2:.4f} | EMA R2={r2_ema:.4f} {best_mark} | "
              f"RMSE={rmse:.4f} | MAE={mae:.4f} | ETA={eta_str}")

        if early_counter >= PATIENCE:
            print(f"⏹ EarlyStopping Triggered for Region {region} (no best update in {PATIENCE} epochs).")
            break

    # --- 최고 R2를 가진 2차 Weight Step 적용 ---
    best_ckpt_for_region = ckpt_best if os.path.exists(ckpt_best) else ckpt_last
    best_region_r2 = -np.inf
    if SECONDARY_WEIGHT_STEPS:
        for p in plants_in_region:
            for w in SECONDARY_WEIGHT_STEPS:
                weight_ckpt_path = os.path.join(region_dir,f"{p}_2nd_w{w}.pt")
                if os.path.exists(weight_ckpt_path):
                    ck = torch.load(weight_ckpt_path, map_location=DEVICE)
                    model.load_state_dict(ck["model_state"])
                    val_preds_list, val_true_list = [], []
                    with torch.no_grad():
                        for Xv, yv in val_loader:
                            Xv, yv = Xv.to(DEVICE), yv.to(DEVICE)
                            yp = model(Xv)
                            val_preds_list.append(yp[:,0].cpu().numpy())
                            val_true_list.append(yv[:,0].cpu().numpy())
                    val_preds_final = np.concatenate(val_preds_list)
                    val_true_final = np.concatenate(val_true_list)
                    r2_curr, _, _ = calc_metrics(val_true_final,val_preds_final)
                    print(f"🌱 Plant={p} | Weight Step={w} | Validation R2={r2_curr:.4f}")
                    if r2_curr > best_region_r2:
                        best_region_r2 = r2_curr
                        best_ckpt_for_region = weight_ckpt_path
                        final_status = f"2차 Step w{w}"

    # --- 최종 Region 평가 ---
    ck = torch.load(best_ckpt_for_region,map_location=DEVICE)
    model.load_state_dict(ck["model_state"])
    val_preds_list, val_true_list = [], []
    with torch.no_grad():
        for Xv, yv in val_loader:
            Xv, yv = Xv.to(DEVICE), yv.to(DEVICE)
            yp = model(Xv)
            val_preds_list.append(yp[:,0].cpu().numpy())
            val_true_list.append(yv[:,0].cpu().numpy())
    val_preds_final = np.concatenate(val_preds_list)
    val_true_final = np.concatenate(val_true_list)
    r2_final, rmse_final, mae_final = calc_metrics(val_true_final,val_preds_final)
    print(f"✅ 최종 평가 완료 (Region={region}) | R2={r2_final:.4f} | RMSE={rmse_final:.4f} | MAE={mae_final:.4f} | "
          f"최종 ckpt={os.path.basename(best_ckpt_for_region)} | 상태={final_status}")

    region_models[region] = best_ckpt_for_region
    region_metrics[region] = (r2_final,rmse_final,mae_final)
    region_ckpt_status[region] = final_status

# ===============================
# 🔹 안전모드 SHAP 분석
# ===============================
def safe_shap_analysis(model, df, features, seq_len=SEQ_LEN, horizon=0, sample_size=SHAP_SAMPLE):
    """
    CNN-BiLSTM 모델에 맞춘 안전한 SHAP 분석
    """
    try:
        # df가 비어있으면 바로 fallback
        if len(df) == 0:
            return pd.DataFrame({"feature": features, "importance":[0]*len(features)})

        X = df[features].values.astype(np.float32)
        if len(X) < seq_len:
            return pd.DataFrame({"feature": features, "importance":[0]*len(features)})

        # sliding window로 시퀀스 생성
        X_seq = np.array([X[i:i+seq_len] for i in range(len(X)-seq_len+1)], dtype=np.float32)

        # SHAP background 샘플링
        bg = shap.sample(X_seq, max(1, min(len(X_seq), int(sample_size/2))))

        # 모델 예측 함수 정의
        def f(x):
            if not isinstance(model, nn.Module):
                return np.zeros((x.shape[0],))
            xt = torch.tensor(x.astype(np.float32)).to(DEVICE)
            with torch.no_grad():
                y = model(xt)
                if y.ndim > 1:
                    return y[:, horizon].cpu().numpy()
                return y.cpu().numpy()

        # KernelExplainer 생성
        explainer = shap.KernelExplainer(f, bg)
        shap_values = explainer.shap_values(X_seq, nsamples=50)

        arr = shap_values[0] if isinstance(shap_values, list) else shap_values
        importance = np.mean(np.abs(arr), axis=0)

        if importance.ndim > 1:
            importance = importance[0]

        return pd.DataFrame({"feature": features, "importance": importance}).sort_values(
            by="importance", ascending=False
        )

    except Exception as e:
        print(f"❌ SHAP 실패: {e}")
        return pd.DataFrame({"feature": features, "importance":[0]*len(features)})

# ===============================
# 🔹 Plant Test Summary + 2차 Weight 적용 + SHAP 통합
# ===============================

summary_rows = []
excluded_plants = []

for plant in train_raw[GROUP_COL].unique():
    ckpt_path = plant_models.get(plant)
    df_test_plant = test_raw[test_raw[GROUP_COL]==plant]

    # 테스트 데이터 없음
    if len(df_test_plant) == 0:
        excluded_plants.append(plant)
        summary_rows.append({
            "plant": plant,
            "weight": "N/A",
            "R2_1차": 0.0,
            "RMSE_1차": float("nan"),
            "MAE_1차": float("nan"),
            "R2_2차": 0.0,
            "RMSE_2차": float("nan"),
            "MAE_2차": float("nan"),
            "improved": False,
            "weight_status": "no_test_data"
        })
        continue

    # 체크포인트 없음
    if not ckpt_path or not os.path.exists(ckpt_path):
        excluded_plants.append(plant)
        summary_rows.append({
            "plant": plant,
            "weight": "no_ckpt",
            "R2_1차": 0.0,
            "RMSE_1차": float("nan"),
            "MAE_1차": float("nan"),
            "R2_2차": 0.0,
            "RMSE_2차": float("nan"),
            "MAE_2차": float("nan"),
            "improved": False,
            "weight_status": "no_ckpt"
        })
        continue

    # 1차 모델 로드
    model = CNN_BiLSTM().to(DEVICE)
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt.get("model_state", ckpt))

    # 1차 예측
    test_preds_1, test_true = predict(model, df_test_plant)
    r2_1, rmse_1, mae_1 = calc_metrics(test_true, test_preds_1)

    # =====================
    # 2차 Weight 적용
    # =====================
    weight_status = "1차 Only"
    r2_2, rmse_2, mae_2 = r2_1, rmse_1, mae_1
    region = None
    weight_applied = None

    if 'region_plant_map' in globals():
        for r, plants in region_plant_map.items():
            if plant in plants:
                region = r
                break

    if region:
        region_dir = os.path.join(CKPT_DIR, "region", region)

        # 2차 Weight Step 존재 시 적용
        for w in SECONDARY_WEIGHT_STEPS:
            weight_ckpt_path = os.path.join(region_dir, f"{plant}_2nd_w{w}.pt")
            if os.path.exists(weight_ckpt_path):
                ck2 = torch.load(weight_ckpt_path, map_location=DEVICE)
                model.load_state_dict(ck2.get("model_state", ck2))
                weight_status = f"2차 Step w{w}"
                weight_applied = w
                break

        # 2차 Weight Step 없으면 Region 최종 ckpt 적용
        if weight_applied is None and region in region_models:
            ck_region = torch.load(region_models[region], map_location=DEVICE)
            model.load_state_dict(ck_region.get("model_state", ck_region))
            weight_status = "Region 최종"

        # 2차 예측
        test_preds_2, _ = predict(model, df_test_plant)
        r2_2, rmse_2, mae_2 = calc_metrics(test_true, test_preds_2)

    improved = r2_2 > r2_1
    summary_rows.append({
        "plant": plant,
        "weight": weight_applied if weight_applied is not None else 1.0,
        "R2_1차": r2_1,
        "RMSE_1차": rmse_1,
        "MAE_1차": mae_1,
        "R2_2차": r2_2,
        "RMSE_2차": rmse_2,
        "MAE_2차": mae_2,
        "improved": improved,
        "weight_status": weight_status
    })

# DataFrame으로 변환 후 CSV 저장
df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv("plant_test_summary.csv", index=False)
print(f"✅ Plant Test Summary 생성 완료 | 총 {len(df_summary)}개 Plant")

# =====================
# HTML 생성
# =====================

html_dashboard_path = os.path.join(SAVE_DIR, "Plant_Test_Summary.html")
with open(html_dashboard_path, "w", encoding="utf-8") as f:
    f.write("<html><head><meta charset='utf-8'><title>Plant & Region Trend Dashboard</title></head><body>")
    f.write("<h1>🌱 Plant & Region Trend Dashboard</h1>")

    # =====================
    # Region별 요약
    # =====================
    f.write("<h2>🌿 Region별 요약</h2>")
    f.write("<table border='1' cellpadding='6'><tr><th>Region</th><th>Avg R2 1차</th><th>Avg R2 2차</th>"
            "<th>Avg RMSE 1차</th><th>Avg RMSE 2차</th>"
            "<th>Avg MAE 1차</th><th>Avg MAE 2차</th></tr>")
    
    for region, plants in region_plant_map.items():
        region_df = df_summary[df_summary['plant'].isin(plants) & (df_summary['weight_status']!="no_test_data")]
        def avg(col): return region_df[col].dropna().mean() if len(region_df)>0 else 0.0
        f.write(f"<tr><td>{region}</td><td>{avg('R2_1차'):.4f}</td><td>{avg('R2_2차'):.4f}</td>"
                f"<td>{avg('RMSE_1차'):.4f}</td><td>{avg('RMSE_2차'):.4f}</td>"
                f"<td>{avg('MAE_1차'):.4f}</td><td>{avg('MAE_2차'):.4f}</td></tr>")
    f.write("</table><br>")

    # =====================
    # Region별 R² 트렌드 그래프
    # =====================
    f.write("<h2>🌿 Region별 Plant R² 트렌드</h2>")
    for region, plants in region_plant_map.items():
        region_df = df_summary[df_summary['plant'].isin(plants)]
        plt.figure(figsize=(10,4))
        x = np.arange(len(plants))
        plt.plot(x, region_df['R2_1차'], marker='o', label='1차 R2', color='skyblue')
        plt.plot(x, region_df['R2_2차'], marker='o', label='2차 R2', color='limegreen')

        # 이상치 강조 (2차 R2 < 0.5)
        for idx, r2_2 in enumerate(region_df['R2_2차']):
            if r2_2 < 0.5:
                plt.scatter(x[idx], r2_2, color='red', s=100, marker='x')
        plt.xticks(x, plants, rotation=45, ha='right')
        plt.ylabel("R²")
        plt.title(f"{region} Region R² Trend (빨강: 2차 R² < 0.5)")
        plt.legend()
        plt.tight_layout()
        buf = BytesIO()
        plt.savefig(buf, format='png')
        plt.close()
        buf.seek(0)
        img_b64 = base64.b64encode(buf.read()).decode('utf-8')
        f.write(f"<img src='data:image/png;base64,{img_b64}' width='900'><br>")

    # =====================
    # Plant별 R² 비교 + SHAP
    # =====================
    f.write("<h2>🌱 Plant별 상세 & SHAP</h2>")
    f.write("<table border='1' cellpadding='6'><tr><th>Plant</th><th>Weight Status</th><th>Weight</th>"
            "<th>R2 1차</th><th>RMSE 1차</th><th>MAE 1차</th>"
            "<th>R2 2차</th><th>RMSE 2차</th><th>MAE 2차</th>"
            "<th>상태</th><th>Top Features (SHAP)</th></tr>")

    for _, row in df_summary.iterrows():
        plant = row['plant']
        status = row.get("weight_status","ok")
        # 색상 설정
        if row.get("improved", False):
            color = " style='background-color:#c6efce;'"  # 개선
        elif status == "no_test_data" or status == "no_ckpt":
            color = " style='background-color:#f0f0f0;color:#666;'"  # 테스트 없음
        elif row['R2_2차'] < 0.5:
            color = " style='background-color:#f8d7da;'"  # 이상치
        else:
            color = ""

        # SHAP 계산
        top_features = "테스트 없음"
        img_html = ""
        df_test_plant = test_raw[test_raw[GROUP_COL]==plant]
        ckpt_path = plant_models.get(plant)

        if len(df_test_plant) > 0 and ckpt_path and os.path.exists(ckpt_path):
            try:
                model = CNN_BiLSTM().to(DEVICE)
                ckpt = torch.load(ckpt_path, map_location=DEVICE)
                model.load_state_dict(ckpt.get("model_state", ckpt))
                shap_df = safe_shap_analysis(model, df_test_plant, features=feature_cols)
                top_features = ", ".join(shap_df['feature'].head(3).tolist())

                # SHAP 막대그래프
                plt.figure(figsize=(6,3))
                plt.bar(shap_df['feature'], shap_df['importance'], color='skyblue')
                plt.xticks(rotation=45, ha='right')
                plt.title(f"{plant} SHAP Importance")
                plt.tight_layout()
                buf = BytesIO()
                plt.savefig(buf, format='png')
                plt.close()
                buf.seek(0)
                img_b64 = base64.b64encode(buf.read()).decode('utf-8')
                img_html = f"<br><img src='data:image/png;base64,{img_b64}' width='300'>"
            except Exception as e:
                top_features += f" (SHAP 실패: {e})"

        f.write(f"<tr{color}><td>{plant}</td><td>{row['weight_status']}</td><td>{row['weight']}</td>"
                f"<td>{safe_fmt(row['R2_1차'])}</td><td>{safe_fmt(row['RMSE_1차'])}</td><td>{safe_fmt(row['MAE_1차'])}</td>"
                f"<td>{safe_fmt(row['R2_2차'])}</td><td>{safe_fmt(row['RMSE_2차'])}</td><td>{safe_fmt(row['MAE_2차'])}</td>"
                f"<td>{status}</td><td>{top_features}{img_html}</td></tr>")

    f.write("</table><br>")

    if excluded_plants:
        f.write("<h2>⚠ 테스트 제외 Plant</h2><ul>")
        for p in excluded_plants: f.write(f"<li>{p}</li>")
        f.write("</ul>")

    f.write("</body></html>")

try:
    import webbrowser
    webbrowser.open(f"file://{html_dashboard_path}")
except:
    pass
print("✅ Plant Test Summary HTML 생성 완료!")

print("🎉 전체 파이프라인 완료!")


✅ Device: cuda
✅ feature_cols (22): ['기온(°C)', '강수량(mm)', '풍속(m/s)', '습도(%)', '증기압(hPa)', '일조(hr)', '일사(MJ/m2)', '적설(cm)', '전운량(10분위)', '중하층운량(10분위)', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'lag_1', 'lag_3', 'lag_6', 'lag_24', 'diff_1', 'diff_3', 'diff_6', 'diff_24']

🌱 Plant 학습 시작: 남제주소내 | train size: 78888
🔄 Checkpoint 로드: C:\ESG_Project1\cnn_lstm\output\checkpoints\plant\남제주소내\남제주소내_best.pt | resume epoch=27 | best_r2=0.9142
Epoch 27/50 | TrainLoss=0.179417 | Val R2=0.9113 | EMA R2=0.9113  | RMSE=0.2634 | MAE=0.1471 | ETA=5.92m
⏹ EarlyStopping Triggered for 남제주소내
✅ 최종 평가 완료 (Plant=남제주소내) | R2=0.9142 | RMSE=0.2591 | MAE=0.1481

🌱 Plant 학습 시작: 부산복합자재창고 | train size: 78888
🔄 Checkpoint 로드: C:\ESG_Project1\cnn_lstm\output\checkpoints\plant\부산복합자재창고\부산복합자재창고_best.pt | resume epoch=33 | best_r2=0.8664
Epoch 33/50 | TrainLoss=0.012081 | Val R2=0.7986 | EMA R2=0.7986  | RMSE=0.1521 | MAE=0.0936 | ETA=4.40m
⏹ EarlyStopping Triggered for 부산복합자재창고
✅ 최종 평가 완료 (Plant=부산복합자재창고) | R2=0.8664

In [ ]:
# ==============================
# HTML 리포트 저장용 helper
# ==============================
html_report = []

def fig_to_html(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    img_b64 = base64.b64encode(buf.read()).decode('utf-8')
    return f'<img src="data:image/png;base64,{img_b64}"/>'

# ==============================
# 3. 이상치 탐지 & 원인 분석
# ==============================
def outlier_analysis_html(model, X_test, y_test, plant_col='plant', datetime_col='datetime'):
    # SHAP explainer
    explainer = shap.Explainer(model, X_test)
    shap_values = explainer(X_test)

    # Residual 계산
    residuals = np.abs(y_test - model.predict(X_test))
    threshold = np.percentile(residuals, 99)  # 상위 1% 이상치
    outlier_idx = np.where(residuals >= threshold)[0]

    outlier_data = X_test.iloc[outlier_idx]

    # 전체 이상치 통계 및 시계열
    total_outliers = len(outlier_idx)
    total_ratio = total_outliers / len(X_test) * 100

    fig, ax = plt.subplots(figsize=(12,4))
    ax.plot(X_test[datetime_col], residuals, label='Residuals')
    ax.scatter(X_test.iloc[outlier_idx][datetime_col], residuals[outlier_idx], color='red', label='Outliers')
    ax.set_xlabel("Time")
    ax.set_ylabel("Residuals")
    ax.set_title(f"전체 이상치 시계열 (Count: {total_outliers}, Ratio: {total_ratio:.2f}%)")
    ax.legend()
    html_report.append(fig_to_html(fig))

    # 발전소별 이상치 통계
    plant_stats = pd.DataFrame({'plant': X_test[plant_col], 'is_outlier': 0})
    plant_stats['is_outlier'] = plant_stats.index.isin(outlier_idx).astype(int)
    plant_stats = plant_stats.groupby('plant')['is_outlier'].agg(['sum','count']).rename(columns={'sum':'outlier_count','count':'total_count'})
    plant_stats['outlier_ratio'] = plant_stats['outlier_count']/plant_stats['total_count']*100
    html_report.append(f"<h3>발전소별 이상치 통계</h3>{plant_stats.to_html()}")

    # 발전소별 시계열 시각화
    for plant, df_plant in X_test.groupby(plant_col):
        plant_outlier_idx = df_plant.index.intersection(outlier_idx)
        if len(plant_outlier_idx)==0:
            continue
        fig, ax = plt.subplots(figsize=(12,4))
        ax.plot(df_plant[datetime_col], residuals[df_plant.index], label='Residuals')
        ax.scatter(df_plant.loc[plant_outlier_idx, datetime_col], residuals[plant_outlier_idx], color='red', label='Outliers')
        ax.set_xlabel("Time")
        ax.set_ylabel("Residuals")
        ax.set_title(f"{plant} 이상치 시계열")
        ax.legend()
        html_report.append(fig_to_html(fig))

    # Top 3 이상치 원인 feature
    outlier_shap_values = shap_values[outlier_idx]
    shap_mean = np.abs(outlier_shap_values.values).mean(axis=0)
    top3_idx = np.argsort(shap_mean)[-3:]
    top3_features = X_test.columns[top3_idx]

    fig, ax = plt.subplots(figsize=(6,4))
    sns.barplot(x=shap_mean[top3_idx], y=top3_features, ax=ax)
    ax.set_title("Top 3 Features Causing Outliers")
    html_report.append(fig_to_html(fig))

# ==============================
# 4. Region별 이상치 분석 실행
# ==============================
for region, model in region_models.items():
    html_report.append(f"<h2>Region 이상치 분석: {region}</h2>")
    df_test_region = test_raw[test_raw['region']==region]
    X_test_region = df_test_region.drop(columns=['target'])
    y_test_region = df_test_region['target']
    outlier_analysis_html(model, X_test_region, y_test_region, plant_col='plant', datetime_col='datetime')

# ==============================
# 5. HTML 리포트 저장
# ==============================
report_path = "outlier_report.html"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("<html><head><meta charset='utf-8'><title>Outlier Report</title></head><body>")
    f.write("\n".join(html_report))
    f.write("</body></html>")

print(f"✅ HTML 리포트 생성 완료: {report_path}")

In [ ]:
# -*- coding: utf-8 -*-
"""
완전 통합 파이프라인 (Plant -> conditional 2nd -> Region)
- CNN_BiLSTM
- 1차 학습 / 조건부 2차 재학습 (가중치 스케일 [1.0,0.5,0.3])
- Plant별 Train/Val R² gap 기반 Fine-tune 전략 자동 선택
- Resume / Checkpoint 안정화 (AMP + optimizer state 포함)
- SHAP 안전분석
- HTML 리포트 (1차/2차 지표 포함)
- 출력 강화
"""
import os, time, warnings, base64, webbrowser, json
from io import BytesIO
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from jinja2 import Template
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import shap

# ----------------------------
# === 사용자 설정 ===
# ----------------------------
TRAIN_CSV = r"C:\ESG_Project1\file\merge_data\train.csv"
VAL_CSV   = r"C:\ESG_Project1\file\merge_data\val.csv"
TEST_CSV  = r"C:\ESG_Project1\file\merge_data\test.csv"
SAVE_DIR  = r"C:\ESG_Project1\cnn_lstm\output"
os.makedirs(SAVE_DIR, exist_ok=True)

TIME_COL, GROUP_COL, REGION_COL, TARGET_COL = "일시", "발전구분", "지역", "합산발전량(MWh)"
WEATHER_COLS = [
    "기온(°C)", "강수량(mm)", "풍속(m/s)", "습도(%)", "증기압(hPa)",
    "일조(hr)", "일사(MJ/m2)", "적설(cm)", "전운량(10분위)", "중하층운량(10분위)"
]
TIME_FEATS = ["hour_sin", "hour_cos", "doy_sin", "doy_cos"]

SEQ_LEN, HORIZON = 168, 24
BATCH = 128
EPOCHS = 50
LR = 1e-3
PATIENCE = 5
SHAP_SAMPLE = 50
VAL_R2_THRESHOLD = 0.9
SECONDARY_WEIGHT_STEPS = [1.0, 0.5, 0.3]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = (DEVICE.type == "cuda")
torch.set_num_threads(max(1, min(4, os.cpu_count()//2)))
print(f"✅ Device: {DEVICE} | AMP: {use_amp}")
warnings.filterwarnings("ignore")
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# ----------------------------
# === 데이터 로드 + 전처리 ===
# ----------------------------
train_raw = pd.read_csv(TRAIN_CSV, parse_dates=[TIME_COL])
val_raw   = pd.read_csv(VAL_CSV, parse_dates=[TIME_COL])
test_raw  = pd.read_csv(TEST_CSV, parse_dates=[TIME_COL])

def add_time_feats(df):
    df = df.copy()
    df.sort_values([GROUP_COL, TIME_COL], inplace=True)
    df["hour"] = df[TIME_COL].dt.hour
    df["doy"] = df[TIME_COL].dt.dayofyear
    df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)
    df["doy_sin"] = np.sin(2*np.pi*df["doy"]/365)
    df["doy_cos"] = np.cos(2*np.pi*df["doy"]/365)
    return df

for df in (train_raw, val_raw, test_raw):
    df[:] = add_time_feats(df)

LAGS = [1,3,6,24]
def add_lag_diff(df):
    df = df.copy()
    for lag in LAGS:
        df[f"lag_{lag}"] = df.groupby(GROUP_COL)[TARGET_COL].shift(lag)
        df[f"diff_{lag}"] = df[TARGET_COL] - df[f"lag_{lag}"]
    df.fillna(0, inplace=True)
    return df

train_raw = add_lag_diff(train_raw)
val_raw   = add_lag_diff(val_raw)
test_raw  = add_lag_diff(test_raw)

all_candidate_feats = WEATHER_COLS + TIME_FEATS + [f"lag_{l}" for l in LAGS] + [f"diff_{l}" for l in LAGS]
feature_cols = [c for c in all_candidate_feats if c in train_raw.columns]
print(f"✅ feature_cols ({len(feature_cols)}): {feature_cols}")

# ----------------------------
# === 지역별 스케일러 적용 ===
# ----------------------------
region_scalers = {}
for region, grp in train_raw.groupby(REGION_COL):
    valid_cols = [c for c in feature_cols if c in grp.columns]
    if len(valid_cols)==0: continue
    std = StandardScaler().fit(grp[valid_cols])
    mm  = MinMaxScaler().fit(std.transform(grp[valid_cols]))
    tsc = StandardScaler().fit(np.log1p(grp[[TARGET_COL]].clip(lower=0.0)+1e-8))
    region_scalers[region] = (std, mm, tsc)
    train_raw.loc[grp.index, valid_cols] = mm.transform(std.transform(grp[valid_cols]))
    train_raw.loc[grp.index, TARGET_COL] = tsc.transform(np.log1p(grp[[TARGET_COL]].clip(lower=0.0)+1e-8))

for df in (val_raw, test_raw):
    for region, grp in df.groupby(REGION_COL):
        valid_cols = [c for c in feature_cols if c in grp.columns]
        if region in region_scalers and len(valid_cols)>0:
            std, mm, tsc = region_scalers[region]
            df.loc[grp.index, valid_cols] = mm.transform(std.transform(grp[valid_cols]))
            df.loc[grp.index, TARGET_COL] = tsc.transform(np.log1p(grp[[TARGET_COL]].clip(lower=0.0)+1e-8))

# ----------------------------
# === Dataset & Model ===
# ----------------------------
class TimeSeriesSeqDataset(Dataset):
    def __init__(self, df, seq_len=SEQ_LEN, horizon=HORIZON, feature_cols=feature_cols):
        self.seq_len = seq_len
        self.horizon = horizon
        self.features = df[feature_cols].values.astype(np.float32)
        self.targets = df[TARGET_COL].values.astype(np.float32)
        self.n_windows = max(0, len(df)-seq_len-horizon+1)
    def __len__(self):
        return self.n_windows
    def __getitem__(self, i):
        x = self.features[i:i+self.seq_len]
        y = self.targets[i+self.seq_len:i+self.seq_len+self.horizon]
        return torch.from_numpy(x), torch.from_numpy(y)

class CNN_BiLSTM(nn.Module):
    def __init__(self, input_dim=len(feature_cols), hidden=128, num_layers=2, horizon=HORIZON):
        super().__init__()
        self.conv1 = nn.Conv1d(input_dim, 128, 3, padding=1)
        self.conv2 = nn.Conv1d(128, 64, 3, padding=1)
        self.lstm = nn.LSTM(64, hidden, num_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden*2, horizon)
        self.relu = nn.ReLU()
        self.norm = nn.LayerNorm(hidden*2)
    def forward(self, x):
        if x.dim()==2: x = x.unsqueeze(0)
        x = x.permute(0,2,1)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = x.permute(0,2,1)
        x,_ = self.lstm(x)
        x = self.norm(x[:,-1,:])
        return self.fc(x)

# ----------------------------
# === 학습/평가 유틸 함수 ===
# ----------------------------
def calc_metrics(y_true, y_pred):
    if len(y_true)==0: return float("nan"), float("nan"), float("nan")
    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    return r2, rmse, mae

def predict(model, df, features=feature_cols):
    ds = TimeSeriesSeqDataset(df, feature_cols=features)
    if len(ds)==0: return np.array([])
    loader = DataLoader(ds, batch_size=BATCH, shuffle=False)
    preds=[]
    model.eval()
    with torch.no_grad():
        for X,_ in loader:
            X=X.to(DEVICE)
            y_pred = model(X)[:,0].cpu().numpy()
            preds.append(y_pred)
    return np.concatenate(preds) if preds else np.array([])

def safe_shap_analysis(model, df, features, sample_size=SHAP_SAMPLE):
    try:
        X = df[features].values.astype(np.float32)
        if len(X)==0: return pd.DataFrame({"feature": features, "importance":[0]*len(features)})
        if len(X)>sample_size: X = X[:sample_size]
        explainer = shap.Explainer(model, X)
        shap_values = explainer(X)
        mean_imp = np.abs(shap_values.values).mean(axis=0)
        return pd.DataFrame({"feature":features, "importance":mean_imp}).sort_values("importance",ascending=False)
    except:
        return pd.DataFrame({"feature": features, "importance":[0]*len(features)})

# ----------------------------
# === Plant별 학습 + Fine-tune 전략 선택 ===
# ----------------------------
plant_models = {}
history = {}
for plant, grp in train_raw.groupby(GROUP_COL):
    print(f"\n🔥 학습 시작: {plant} | len={len(grp)}")
    model = CNN_BiLSTM().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.MSELoss()
    best_r2, wait = -np.inf, 0
    train_ds = TimeSeriesSeqDataset(grp)
    val_grp = val_raw[val_raw[GROUP_COL]==plant]
    val_ds   = TimeSeriesSeqDataset(val_grp)
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH, shuffle=False)

    scaler = region_scalers.get(grp[REGION_COL].iloc[0], None)
    for epoch in range(EPOCHS):
        model.train()
        running_loss=0
        for X,y in train_loader:
            X,y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=use_amp):
                y_pred = model(X)
                loss = criterion(y_pred, y)
            if use_amp:
                scaler_ = torch.cuda.amp.GradScaler()
                scaler_.scale(loss).backward()
                scaler_.step(optimizer)
                scaler_.update()
            else:
                loss.backward()
                optimizer.step()
            running_loss += loss.item()*len(X)
        running_loss /= max(1,len(train_ds))
        
        # Validation
        model.eval()
        val_preds=[]
        val_trues=[]
        with torch.no_grad():
            for Xv, yv in val_loader:
                Xv,yv = Xv.to(DEVICE), yv.to(DEVICE)
                y_pred = model(Xv)
                val_preds.append(y_pred.cpu().numpy())
                val_trues.append(yv.cpu().numpy())
        val_preds = np.concatenate(val_preds) if val_preds else np.array([])
        val_trues = np.concatenate(val_trues) if val_trues else np.array([])
        r2, rmse, mae = calc_metrics(val_trues, val_preds)

        print(f"Epoch {epoch+1}/{EPOCHS} | Loss={running_loss:.5f} | Val R2={r2:.4f} | MAE={mae:.4f} | RMSE={rmse:.4f}")
        # Best model save
        if r2 > best_r2:
            best_r2 = r2
            wait=0
            ckpt_path = os.path.join(SAVE_DIR, f"{plant}_best.pth")
            torch.save(model.state_dict(), ckpt_path)
        else:
            wait+=1
            if wait>=PATIENCE: break
    plant_models[plant] = model
    history[plant] = {"best_r2": best_r2}

print("\n✅ Plant별 학습 완료!")

# ----------------------------
# === Plant별 SHAP 및 HTML Report 예시 ===
# ----------------------------
shap_results=[]
for plant, model in plant_models.items():
    grp = train_raw[train_raw[GROUP_COL]==plant]
    shap_df = safe_shap_analysis(model, grp, feature_cols)
    shap_results.append((plant, shap_df))

# HTML report 생성
html_template = """
<html><head><meta charset='utf-8'><title>Plant SHAP Report</title></head><body>
<h1>Plant별 SHAP Top Features</h1>
{% for plant, df in shap_results %}
<h2>{{plant}}</h2>
<table border=1>
<tr><th>Feature</th><th>Importance</th></tr>
{% for idx,row in df.iterrows() %}
<tr><td>{{row.feature}}</td><td>{{row.importance | round(5)}}</td></tr>
{% endfor %}
</table>
{% endfor %}
</body></html>
"""
report_path = os.path.join(SAVE_DIR,"shap_report.html")
with open(report_path,"w",encoding="utf-8") as f:
    f.write(Template(html_template).render(shap_results=shap_results))
webbrowser.open(report_path)
print(f"✅ SHAP HTML Report saved: {report_path}")

# -*- coding: utf-8 -*-
"""
완전 통합 파이프라인 (Plant -> conditional 2nd -> Region)
- CNN_BiLSTM
- 1차 학습 / 조건부 2차 재학습 (가중치 스케일 [1.0,0.5,0.3])
- Plant별 Train/Val R² gap 기반 Fine-tune 전략 자동 선택
- Resume / Checkpoint 안정화 (AMP + optimizer state 포함)
- SHAP 안전분석
- HTML 리포트 (1차/2차 지표 포함)
- 출력 강화
"""
import os, time, warnings, base64, webbrowser, json
from io import BytesIO
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from jinja2 import Template
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import shap

# ----------------------------
# === 사용자 설정 ===
# ----------------------------
TRAIN_CSV = r"C:\ESG_Project1\file\merge_data\train.csv"
VAL_CSV   = r"C:\ESG_Project1\file\merge_data\val.csv"
TEST_CSV  = r"C:\ESG_Project1\file\merge_data\test.csv"
SAVE_DIR  = r"C:\ESG_Project1\cnn_lstm\output"
os.makedirs(SAVE_DIR, exist_ok=True)

TIME_COL, GROUP_COL, REGION_COL, TARGET_COL = "일시", "발전구분", "지역", "합산발전량(MWh)"
WEATHER_COLS = [
    "기온(°C)", "강수량(mm)", "풍속(m/s)", "습도(%)", "증기압(hPa)",
    "일조(hr)", "일사(MJ/m2)", "적설(cm)", "전운량(10분위)", "중하층운량(10분위)"
]
TIME_FEATS = ["hour_sin", "hour_cos", "doy_sin", "doy_cos"]

SEQ_LEN, HORIZON = 168, 24
BATCH = 128
EPOCHS = 50
LR = 1e-3
PATIENCE = 5
SHAP_SAMPLE = 50
VAL_R2_THRESHOLD = 0.9
SECONDARY_WEIGHT_STEPS = [1.0, 0.5, 0.3]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = (DEVICE.type == "cuda")
torch.set_num_threads(max(1, min(4, os.cpu_count()//2)))
print(f"✅ Device: {DEVICE} | AMP: {use_amp}")
warnings.filterwarnings("ignore")
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# ----------------------------
# === 데이터 로드 + 전처리 ===
# ----------------------------
train_raw = pd.read_csv(TRAIN_CSV, parse_dates=[TIME_COL])
val_raw   = pd.read_csv(VAL_CSV, parse_dates=[TIME_COL])
test_raw  = pd.read_csv(TEST_CSV, parse_dates=[TIME_COL])

def add_time_feats(df):
    df = df.copy()
    df.sort_values([GROUP_COL, TIME_COL], inplace=True)
    df["hour"] = df[TIME_COL].dt.hour
    df["doy"] = df[TIME_COL].dt.dayofyear
    df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)
    df["doy_sin"] = np.sin(2*np.pi*df["doy"]/365)
    df["doy_cos"] = np.cos(2*np.pi*df["doy"]/365)
    return df

for df in (train_raw, val_raw, test_raw):
    df[:] = add_time_feats(df)

LAGS = [1,3,6,24]
def add_lag_diff(df):
    df = df.copy()
    for lag in LAGS:
        df[f"lag_{lag}"] = df.groupby(GROUP_COL)[TARGET_COL].shift(lag)
        df[f"diff_{lag}"] = df[TARGET_COL] - df[f"lag_{lag}"]
    df.fillna(0, inplace=True)
    return df

train_raw = add_lag_diff(train_raw)
val_raw   = add_lag_diff(val_raw)
test_raw  = add_lag_diff(test_raw)

all_candidate_feats = WEATHER_COLS + TIME_FEATS + [f"lag_{l}" for l in LAGS] + [f"diff_{l}" for l in LAGS]
feature_cols = [c for c in all_candidate_feats if c in train_raw.columns]
print(f"✅ feature_cols ({len(feature_cols)}): {feature_cols}")

# ----------------------------
# === 지역별 스케일러 적용 ===
# ----------------------------
region_scalers = {}
for region, grp in train_raw.groupby(REGION_COL):
    valid_cols = [c for c in feature_cols if c in grp.columns]
    if len(valid_cols)==0: continue
    std = StandardScaler().fit(grp[valid_cols])
    mm  = MinMaxScaler().fit(std.transform(grp[valid_cols]))
    tsc = StandardScaler().fit(np.log1p(grp[[TARGET_COL]].clip(lower=0.0)+1e-8))
    region_scalers[region] = (std, mm, tsc)
    train_raw.loc[grp.index, valid_cols] = mm.transform(std.transform(grp[valid_cols]))
    train_raw.loc[grp.index, TARGET_COL] = tsc.transform(np.log1p(grp[[TARGET_COL]].clip(lower=0.0)+1e-8))

for df in (val_raw, test_raw):
    for region, grp in df.groupby(REGION_COL):
        valid_cols = [c for c in feature_cols if c in grp.columns]
        if region in region_scalers and len(valid_cols)>0:
            std, mm, tsc = region_scalers[region]
            df.loc[grp.index, valid_cols] = mm.transform(std.transform(grp[valid_cols]))
            df.loc[grp.index, TARGET_COL] = tsc.transform(np.log1p(grp[[TARGET_COL]].clip(lower=0.0)+1e-8))

# ----------------------------
# === Dataset & Model ===
# ----------------------------
class TimeSeriesSeqDataset(Dataset):
    def __init__(self, df, seq_len=SEQ_LEN, horizon=HORIZON, feature_cols=feature_cols):
        self.seq_len = seq_len
        self.horizon = horizon
        self.features = df[feature_cols].values.astype(np.float32)
        self.targets = df[TARGET_COL].values.astype(np.float32)
        self.n_windows = max(0, len(df)-seq_len-horizon+1)
    def __len__(self):
        return self.n_windows
    def __getitem__(self, i):
        x = self.features[i:i+self.seq_len]
        y = self.targets[i+self.seq_len:i+self.seq_len+self.horizon]
        return torch.from_numpy(x), torch.from_numpy(y)

class CNN_BiLSTM(nn.Module):
    def __init__(self, input_dim=len(feature_cols), hidden=128, num_layers=2, horizon=HORIZON):
        super().__init__()
        self.conv1 = nn.Conv1d(input_dim, 128, 3, padding=1)
        self.conv2 = nn.Conv1d(128, 64, 3, padding=1)
        self.lstm = nn.LSTM(64, hidden, num_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden*2, horizon)
        self.relu = nn.ReLU()
        self.norm = nn.LayerNorm(hidden*2)
    def forward(self, x):
        if x.dim()==2: x = x.unsqueeze(0)
        x = x.permute(0,2,1)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = x.permute(0,2,1)
        x,_ = self.lstm(x)
        x = self.norm(x[:,-1,:])
        return self.fc(x)

# ----------------------------
# === 학습/평가 유틸 함수 ===
# ----------------------------
def calc_metrics(y_true, y_pred):
    if len(y_true)==0: return float("nan"), float("nan"), float("nan")
    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    return r2, rmse, mae

def predict(model, df, features=feature_cols):
    ds = TimeSeriesSeqDataset(df, feature_cols=features)
    if len(ds)==0: return np.array([])
    loader = DataLoader(ds, batch_size=BATCH, shuffle=False)
    preds=[]
    model.eval()
    with torch.no_grad():
        for X,_ in loader:
            X=X.to(DEVICE)
            y_pred = model(X)[:,0].cpu().numpy()
            preds.append(y_pred)
    return np.concatenate(preds) if preds else np.array([])

def safe_shap_analysis(model, df, features, sample_size=SHAP_SAMPLE):
    try:
        X = df[features].values.astype(np.float32)
        if len(X)==0: return pd.DataFrame({"feature": features, "importance":[0]*len(features)})
        if len(X)>sample_size: X = X[:sample_size]
        explainer = shap.Explainer(model, X)
        shap_values = explainer(X)
        mean_imp = np.abs(shap_values.values).mean(axis=0)
        return pd.DataFrame({"feature":features, "importance":mean_imp}).sort_values("importance",ascending=False)
    except:
        return pd.DataFrame({"feature": features, "importance":[0]*len(features)})

# ----------------------------
# === Plant별 학습 + Fine-tune 전략 선택 ===
# ----------------------------
plant_models = {}
history = {}
for plant, grp in train_raw.groupby(GROUP_COL):
    print(f"\n🔥 학습 시작: {plant} | len={len(grp)}")
    model = CNN_BiLSTM().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.MSELoss()
    best_r2, wait = -np.inf, 0
    train_ds = TimeSeriesSeqDataset(grp)
    val_grp = val_raw[val_raw[GROUP_COL]==plant]
    val_ds   = TimeSeriesSeqDataset(val_grp)
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH, shuffle=False)

    scaler = region_scalers.get(grp[REGION_COL].iloc[0], None)
    for epoch in range(EPOCHS):
        model.train()
        running_loss=0
        for X,y in train_loader:
            X,y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=use_amp):
                y_pred = model(X)
                loss = criterion(y_pred, y)
            if use_amp:
                scaler_ = torch.cuda.amp.GradScaler()
                scaler_.scale(loss).backward()
                scaler_.step(optimizer)
                scaler_.update()
            else:
                loss.backward()
                optimizer.step()
            running_loss += loss.item()*len(X)
        running_loss /= max(1,len(train_ds))
        
        # Validation
        model.eval()
        val_preds=[]
        val_trues=[]
        with torch.no_grad():
            for Xv, yv in val_loader:
                Xv,yv = Xv.to(DEVICE), yv.to(DEVICE)
                y_pred = model(Xv)
                val_preds.append(y_pred.cpu().numpy())
                val_trues.append(yv.cpu().numpy())
        val_preds = np.concatenate(val_preds) if val_preds else np.array([])
        val_trues = np.concatenate(val_trues) if val_trues else np.array([])
        r2, rmse, mae = calc_metrics(val_trues, val_preds)

        print(f"Epoch {epoch+1}/{EPOCHS} | Loss={running_loss:.5f} | Val R2={r2:.4f} | MAE={mae:.4f} | RMSE={rmse:.4f}")
        # Best model save
        if r2 > best_r2:
            best_r2 = r2
            wait=0
            ckpt_path = os.path.join(SAVE_DIR, f"{plant}_best.pth")
            torch.save(model.state_dict(), ckpt_path)
        else:
            wait+=1
            if wait>=PATIENCE: break
    plant_models[plant] = model
    history[plant] = {"best_r2": best_r2}

print("\n✅ Plant별 학습 완료!")

# ----------------------------
# === Plant별 SHAP 및 HTML Report 예시 ===
# ----------------------------
shap_results=[]
for plant, model in plant_models.items():
    grp = train_raw[train_raw[GROUP_COL]==plant]
    shap_df = safe_shap_analysis(model, grp, feature_cols)
    shap_results.append((plant, shap_df))

# HTML report 생성
html_template = """
<html><head><meta charset='utf-8'><title>Plant SHAP Report</title></head><body>
<h1>Plant별 SHAP Top Features</h1>
{% for plant, df in shap_results %}
<h2>{{plant}}</h2>
<table border=1>
<tr><th>Feature</th><th>Importance</th></tr>
{% for idx,row in df.iterrows() %}
<tr><td>{{row.feature}}</td><td>{{row.importance | round(5)}}</td></tr>
{% endfor %}
</table>
{% endfor %}
</body></html>
"""
report_path = os.path.join(SAVE_DIR,"shap_report.html")
with open(report_path,"w",encoding="utf-8") as f:
    f.write(Template(html_template).render(shap_results=shap_results))
webbrowser.open(report_path)
print(f"✅ SHAP HTML Report saved: {report_path}")
